# Lab 4. Linear Processes, AR, MA, and ARMA Models

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-04-linear-processes-ar-ma-arma-lab.ipynb)

This lab is designed for **independent study**. You should be able to work through it even before class discussion, because the necessary background is included before each programming task.

The goal is not only to run code. The goal is to understand how simple mathematical assumptions create the time-series behavior we see in plots and autocorrelation functions.

## What you will learn

By the end of this lab, you should be able to:

1. Simulate white noise, AR(1), MA(1), and ARMA(1,1) models.
2. Explain the difference between AR, MA, and ARMA dependence.
3. Use the backshift notation as a compact way to write models.
4. Check simple causality and invertibility conditions using roots.
5. Compare sample ACF plots with theoretical ACF behavior.
6. Interpret linear-process weights, also called impulse-response weights.
7. Use AI assistance responsibly to critique time-series simulations and model interpretation.

> Colab note: This notebook avoids calligraphic LaTeX commands because they often display poorly in Google Colab.

## 0. Background: why AR, MA, and ARMA models matter

A time series is a sequence of observations ordered by time. In this course, we usually think of the observed data

$$
x_1, x_2, \ldots, x_n
$$

as one realization of a stochastic process

$$
X_1, X_2, \ldots, X_n.
$$

In ordinary regression or classification, it is common to assume observations are independent. In time series analysis, dependence is the main object of study. The current value may depend on previous values, previous shocks, seasonal patterns, or hidden states.

In this lab, we study the most important classical linear models:

- **White noise**: no linear dependence across time.
- **AR model**: the present depends on previous values of the series.
- **MA model**: the present depends on current and previous noise shocks.
- **ARMA model**: the present depends on both previous values and previous noise shocks.

These models are simple, but they are foundational. They also help us understand modern forecasting models, because many neural and machine-learning forecasting models can be viewed as nonlinear extensions of the same idea: use the past to predict the future.

## 1. Mathematical review

### 1.1 White noise

A white-noise process is usually written

$$
W_t \sim WN(0, \sigma^2).
$$

This means

$$
E(W_t)=0, \qquad Var(W_t)=\sigma^2, \qquad Cov(W_s,W_t)=0 \quad \text{for } s \ne t.
$$

White noise is the basic source of randomness in AR, MA, and ARMA models.

### 1.2 Backshift operator

The backshift operator $B$ shifts a time series one step into the past:

$$
B X_t = X_{t-1}, \qquad B^2 X_t = X_{t-2}.
$$

The backshift notation lets us write models compactly.

### 1.3 AR(1)

An autoregressive model of order 1 is

$$
X_t = \phi X_{t-1} + W_t.
$$

Using the backshift operator,

$$
(1 - \phi B)X_t = W_t.
$$

When $|\phi|<1$, this model is stable and can be written as an infinite moving average:

$$
X_t = W_t + \phi W_{t-1} + \phi^2 W_{t-2}+\cdots.
$$

The weights $1, \phi, \phi^2, \ldots$ show how past shocks affect the present.

### 1.4 MA(1)

A moving-average model of order 1 is

$$
X_t = W_t + \theta W_{t-1}.
$$

Using backshift notation,

$$
X_t = (1 + \theta B)W_t.
$$

The process only depends on the current shock and one previous shock, so its autocorrelation cuts off after lag 1.

### 1.5 ARMA(1,1)

An ARMA(1,1) model is

$$
X_t = \phi X_{t-1} + W_t + \theta W_{t-1}.
$$

Using backshift notation,

$$
(1-\phi B)X_t = (1+\theta B)W_t.
$$

This model combines AR-style persistence and MA-style shock adjustment.

## 2. Setup

Run the next cell first. It imports the libraries we need and defines a few plotting defaults.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(7339)

## 3. Helper functions

We will write our own simulation functions first. This is better for learning than immediately using a package.

The main idea is recursive generation. For an AR(1), once we know $X_{t-1}$ and $W_t$, we can compute $X_t$.

In [ ]:
def simulate_ar1(phi, n=400, burnin=200, sigma=1.0, seed=7339):
    # Simulate X_t = phi X_{t-1} + W_t.
    rng_local = np.random.default_rng(seed)
    total = n + burnin
    w = rng_local.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = phi * x[t - 1] + w[t]
    return x[burnin:], w[burnin:]


def simulate_ma1(theta, n=400, burnin=200, sigma=1.0, seed=7339):
    # Simulate X_t = W_t + theta W_{t-1}.
    rng_local = np.random.default_rng(seed)
    total = n + burnin
    w = rng_local.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = w[t] + theta * w[t - 1]
    return x[burnin:], w[burnin:]


def simulate_arma11(phi, theta, n=400, burnin=200, sigma=1.0, seed=7339):
    # Simulate X_t = phi X_{t-1} + W_t + theta W_{t-1}.
    rng_local = np.random.default_rng(seed)
    total = n + burnin
    w = rng_local.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = phi * x[t - 1] + w[t] + theta * w[t - 1]
    return x[burnin:], w[burnin:]


def sample_acf(x, max_lag=30):
    # Compute sample ACF from scratch using denominator n.
    x = np.asarray(x)
    n = len(x)
    x_centered = x - x.mean()
    gamma0 = np.sum(x_centered * x_centered) / n
    acf_values = []
    for h in range(max_lag + 1):
        gamma_h = np.sum(x_centered[h:] * x_centered[:n-h]) / n
        acf_values.append(gamma_h / gamma0)
    return np.array(acf_values)


def plot_series_and_acf(x, title, max_lag=30):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(x)
    axes[0].set_title(title)
    axes[0].set_xlabel("time")
    axes[0].set_ylabel("value")
    plot_acf(x, ax=axes[1], lags=max_lag, zero=True)
    axes[1].set_title("Sample ACF")
    plt.tight_layout()
    plt.show()

## 4. White noise: the baseline process

White noise has no autocorrelation at nonzero lags. In a finite sample, the sample ACF will not be exactly zero, but most values should be small.

For a white-noise process, roughly speaking, sample autocorrelations at nonzero lags should usually fall inside the approximate bounds

$$
\pm \frac{2}{\sqrt{n}}.
$$

This rule is only approximate, but it is a useful diagnostic.

In [ ]:
n = 400
w = rng.normal(0, 1, size=n)
plot_series_and_acf(w, "Gaussian white noise", max_lag=30)

bound = 2 / np.sqrt(n)
print(f"Approximate white-noise ACF bounds: +/- {bound:.3f}")
print(f"Sample mean: {w.mean():.3f}")
print(f"Sample variance: {w.var(ddof=1):.3f}")

### Checkpoint 1

Answer these questions before continuing:

1. Does the white-noise plot show an obvious trend?
2. Are most sample ACF values inside the approximate bounds?
3. Why should we not expect the sample ACF to be exactly zero?

## 5. AR(1): dependence through previous values

Now simulate

$$
X_t = \phi X_{t-1} + W_t.
$$

For $|\phi|<1$, the process is stable. Its theoretical ACF is

$$
\rho(h) = \phi^h, \qquad h=0,1,2,\ldots.
$$

So the ACF decays geometrically. If $\phi>0$, the decay is positive and smooth. If $\phi<0$, the ACF alternates signs.

In [ ]:
phi = 0.75
x_ar1, _ = simulate_ar1(phi=phi, n=400, burnin=300, seed=10)
plot_series_and_acf(x_ar1, f"Stationary AR(1), phi={phi}", max_lag=30)

acf_ar1 = sample_acf(x_ar1, max_lag=15)
theory_ar1 = np.array([phi**h for h in range(16)])

comparison = pd.DataFrame({
    "lag": np.arange(16),
    "sample_acf": acf_ar1,
    "theoretical_acf": theory_ar1
})
comparison.head(10)

In [ ]:
plt.figure(figsize=(8, 4))
plt.stem(comparison["lag"], comparison["sample_acf"], linefmt="C0-", markerfmt="C0o", basefmt=" ", label="sample ACF")
plt.plot(comparison["lag"], comparison["theoretical_acf"], marker="o", label="theoretical ACF")
plt.axhline(0, linewidth=1)
plt.xlabel("lag")
plt.ylabel("ACF")
plt.title("AR(1): sample ACF versus theoretical ACF")
plt.legend()
plt.show()

### Checkpoint 2

For $\phi=0.75$, answer:

1. Does the ACF decay gradually or cut off quickly?
2. Why is the sample ACF not exactly equal to the theoretical ACF?
3. Would you describe this process as short-memory or persistent?

## 6. Stable versus explosive AR(1)

When $|\phi|<1$, the AR(1) process is stable. When $|\phi|>1$, the recursion amplifies past values and can become explosive.

The next cell compares three values of $\phi$.

In [ ]:
phis = [0.5, -0.7, 1.03]
fig, axes = plt.subplots(len(phis), 2, figsize=(13, 9))

for row, phi_value in enumerate(phis):
    x, _ = simulate_ar1(phi=phi_value, n=250, burnin=50, seed=20)
    axes[row, 0].plot(x)
    axes[row, 0].set_title(f"AR(1) time plot, phi={phi_value}")
    axes[row, 0].set_xlabel("time")
    axes[row, 0].set_ylabel("value")
    plot_acf(x, ax=axes[row, 1], lags=25, zero=True)
    axes[row, 1].set_title(f"ACF, phi={phi_value}")

plt.tight_layout()
plt.show()

### Interpretation note

For $\phi=1.03$, the series can grow very large in magnitude. In practice, such behavior is often a sign that the process is not stationary. A slowly decaying ACF alone does not prove an AR model is appropriate; you should also inspect the time plot.

## 7. MA(1): dependence through previous shocks

Now simulate

$$
X_t = W_t + \theta W_{t-1}.
$$

The theoretical ACF of an MA(1) is

$$
\rho(0)=1, \qquad \rho(1)=\frac{\theta}{1+\theta^2}, \qquad \rho(h)=0 \text{ for } h>1.
$$

So an MA(1) process has an ACF that **cuts off** after lag 1.

In [ ]:
theta = 0.70
x_ma1, _ = simulate_ma1(theta=theta, n=400, burnin=300, seed=30)
plot_series_and_acf(x_ma1, f"MA(1), theta={theta}", max_lag=30)

acf_ma1 = sample_acf(x_ma1, max_lag=10)
theory_ma1 = np.zeros(11)
theory_ma1[0] = 1
theory_ma1[1] = theta / (1 + theta**2)

comparison_ma = pd.DataFrame({
    "lag": np.arange(11),
    "sample_acf": acf_ma1,
    "theoretical_acf": theory_ma1
})
comparison_ma

In [ ]:
plt.figure(figsize=(8, 4))
plt.stem(comparison_ma["lag"], comparison_ma["sample_acf"], linefmt="C0-", markerfmt="C0o", basefmt=" ", label="sample ACF")
plt.plot(comparison_ma["lag"], comparison_ma["theoretical_acf"], marker="o", label="theoretical ACF")
plt.axhline(0, linewidth=1)
plt.xlabel("lag")
plt.ylabel("ACF")
plt.title("MA(1): sample ACF versus theoretical ACF")
plt.legend()
plt.show()

### Checkpoint 3

Compare the AR(1) and MA(1) examples:

1. Which model has a geometrically decaying ACF?
2. Which model has an ACF that cuts off after lag 1?
3. Why does finite-sample noise make the sample MA(1) ACF nonzero at lags larger than 1?

## 8. ARMA(1,1): combining persistence and shocks

Now simulate

$$
X_t = \phi X_{t-1} + W_t + \theta W_{t-1}.
$$

This process combines two mechanisms:

- The AR part $\phi X_{t-1}$ creates persistence from previous values.
- The MA part $\theta W_{t-1}$ adds dependence from previous shocks.

For $|\phi|<1$, the ARMA(1,1) model has a causal infinite moving-average representation.

In [ ]:
phi = 0.65
theta = 0.50
x_arma11, _ = simulate_arma11(phi=phi, theta=theta, n=400, burnin=300, seed=40)
plot_series_and_acf(x_arma11, f"ARMA(1,1), phi={phi}, theta={theta}", max_lag=30)

### Interpretation note

ARMA ACF patterns are usually less obvious than pure AR or pure MA patterns. In many applications, ARMA models are selected by combining:

1. time plot,
2. ACF and PACF,
3. parameter estimation,
4. residual diagnostics,
5. information criteria such as AIC and BIC.

## 9. Linear-process weights: impulse responses

A causal ARMA process can often be written as

$$
X_t = \psi_0 W_t + \psi_1 W_{t-1} + \psi_2 W_{t-2} + \cdots.
$$

The coefficients $\psi_j$ are called linear-process weights or impulse-response weights. They measure how much a shock at time $t-j$ contributes to $X_t$.

For AR(1),

$$
\psi_j = \phi^j.
$$

For ARMA(1,1),

$$
\psi_0=1, \qquad \psi_j=(\phi+\theta)\phi^{j-1} \quad \text{for } j \ge 1.
$$

In [ ]:
max_j = 25
phi = 0.65
theta = 0.50

psi_ar1 = np.array([phi**j for j in range(max_j + 1)])
psi_arma11 = np.zeros(max_j + 1)
psi_arma11[0] = 1
for j in range(1, max_j + 1):
    psi_arma11[j] = (phi + theta) * (phi ** (j - 1))

plt.figure(figsize=(9, 4))
plt.plot(np.arange(max_j + 1), psi_ar1, marker="o", label="AR(1) psi weights")
plt.plot(np.arange(max_j + 1), psi_arma11, marker="o", label="ARMA(1,1) psi weights")
plt.axhline(0, linewidth=1)
plt.xlabel("j")
plt.ylabel("psi_j")
plt.title("Impulse-response weights")
plt.legend()
plt.show()

### Checkpoint 4

1. Do the impulse-response weights decay?
2. What would happen to AR(1) weights if $|\phi|>1$?
3. Why is decay of the weights related to stability?

## 10. Root conditions for causality and invertibility

For an ARMA model

$$
\phi(B)X_t = \theta(B)W_t,
$$

we use two polynomials:

$$
\phi(z) = 1 - \phi_1 z - \cdots - \phi_p z^p,
$$

and

$$
\theta(z) = 1 + \theta_1 z + \cdots + \theta_q z^q.
$$

Rules of thumb:

- The AR part is causal when the roots of $\phi(z)=0$ are outside the unit circle.
- The MA part is invertible when the roots of $\theta(z)=0$ are outside the unit circle.

For AR(1), $1-\phi z=0$, so the root is $z=1/\phi$. The condition $|z|>1$ is equivalent to $|\phi|<1$.

In [ ]:
def ar_roots_from_phi(phi_coeffs):
    # For phi(z)=1 - phi1 z - ... - phip z^p, return roots in z.
    # np.roots expects highest power first.
    coeffs = [-c for c in phi_coeffs[::-1]] + [1]
    return np.roots(coeffs)


def ma_roots_from_theta(theta_coeffs):
    # For theta(z)=1 + theta1 z + ... + thetaq z^q, return roots in z.
    coeffs = list(theta_coeffs[::-1]) + [1]
    return np.roots(coeffs)


def outside_unit_circle(roots, tol=1e-10):
    return np.all(np.abs(roots) > 1 + tol)

# Examples
examples = [
    {"name": "AR(1), phi=0.75", "ar": [0.75], "ma": []},
    {"name": "AR(1), phi=1.10", "ar": [1.10], "ma": []},
    {"name": "MA(1), theta=0.70", "ar": [], "ma": [0.70]},
    {"name": "MA(1), theta=1.40", "ar": [], "ma": [1.40]},
    {"name": "ARMA(1,1), phi=0.65, theta=0.50", "ar": [0.65], "ma": [0.50]},
]

rows = []
for ex in examples:
    ar_roots = ar_roots_from_phi(ex["ar"]) if ex["ar"] else np.array([])
    ma_roots = ma_roots_from_theta(ex["ma"]) if ex["ma"] else np.array([])
    rows.append({
        "model": ex["name"],
        "AR roots": np.round(ar_roots, 3),
        "MA roots": np.round(ma_roots, 3),
        "causal_AR_part": outside_unit_circle(ar_roots) if len(ar_roots) else "not applicable",
        "invertible_MA_part": outside_unit_circle(ma_roots) if len(ma_roots) else "not applicable",
    })

pd.DataFrame(rows)

## 11. Visualizing roots

The next cell plots roots relative to the unit circle. Roots outside the unit circle indicate good behavior for the corresponding AR or MA part.

In [ ]:
def plot_roots(ar_coeffs=None, ma_coeffs=None, title="Root plot"):
    ar_coeffs = ar_coeffs or []
    ma_coeffs = ma_coeffs or []
    ar_roots = ar_roots_from_phi(ar_coeffs) if ar_coeffs else np.array([])
    ma_roots = ma_roots_from_theta(ma_coeffs) if ma_coeffs else np.array([])

    theta_grid = np.linspace(0, 2*np.pi, 400)
    plt.figure(figsize=(6, 6))
    plt.plot(np.cos(theta_grid), np.sin(theta_grid), linestyle="--", label="unit circle")

    if len(ar_roots):
        plt.scatter(ar_roots.real, ar_roots.imag, s=90, marker="o", label="AR roots")
    if len(ma_roots):
        plt.scatter(ma_roots.real, ma_roots.imag, s=90, marker="x", label="MA roots")

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlabel("real part")
    plt.ylabel("imaginary part")
    plt.title(title)
    plt.axis("equal")
    plt.xlim(-3, 3)
    plt.ylim(-3, 3)
    plt.legend()
    plt.show()

plot_roots(ar_coeffs=[0.65], ma_coeffs=[0.50], title="ARMA(1,1): phi=0.65, theta=0.50")
plot_roots(ar_coeffs=[1.10], ma_coeffs=[], title="AR(1): phi=1.10")

### Checkpoint 5

1. For AR(1) with $\phi=0.65$, where is the AR root?
2. For AR(1) with $\phi=1.10$, why is the model not causal?
3. For MA(1), what does invertibility mean in terms of the root of $1+\theta z$?

## 12. Model signatures: ACF and PACF

A useful first diagnostic is the pattern of the ACF and PACF.

Typical idealized patterns:

- AR(p): ACF decays; PACF cuts off after lag $p$.
- MA(q): ACF cuts off after lag $q$; PACF decays.
- ARMA(p,q): both ACF and PACF usually decay.

These are not automatic rules. In real data, finite-sample noise, trends, seasonality, and outliers can make the pattern ambiguous.

In [ ]:
def plot_acf_pacf(x, title, lags=30):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf(x, ax=axes[0], lags=lags, zero=True)
    axes[0].set_title(title + ": ACF")
    plot_pacf(x, ax=axes[1], lags=lags, zero=True, method="ywm")
    axes[1].set_title(title + ": PACF")
    plt.tight_layout()
    plt.show()

plot_acf_pacf(x_ar1, "AR(1)", lags=30)
plot_acf_pacf(x_ma1, "MA(1)", lags=30)
plot_acf_pacf(x_arma11, "ARMA(1,1)", lags=30)

## 13. Optional preview: fitting ARMA models with software

Later chapters discuss estimation and diagnostics more carefully. Here we do a short preview using `statsmodels`.

We simulate an ARMA(1,1) process and fit an ARIMA model with order `(1,0,1)`. The middle value 0 means no differencing, so this is an ARMA model.

In [ ]:
fit_model = ARIMA(x_arma11, order=(1, 0, 1), trend="n")
fit_result = fit_model.fit()
print(fit_result.summary())

### Interpretation note

Because we generated the data with $\phi=0.65$ and $\theta=0.50$, the fitted AR and MA estimates should be in the same general neighborhood. They will not match exactly because the data are random and the sample size is finite.

## 14. Independent practice: identify the model type

The next cell creates three unlabeled series. For each one, inspect the time plot, ACF, and PACF. Try to decide whether it looks more like AR(1), MA(1), or ARMA(1,1).

Do not look at the answer cell until you have made your own guesses.

In [ ]:
unknown = {}
unknown["Series A"], _ = simulate_ma1(theta=-0.65, n=400, burnin=300, seed=101)
unknown["Series B"], _ = simulate_ar1(phi=0.80, n=400, burnin=300, seed=102)
unknown["Series C"], _ = simulate_arma11(phi=0.55, theta=0.70, n=400, burnin=300, seed=103)

for name, values in unknown.items():
    plot_series_and_acf(values, name, max_lag=30)
    plot_acf_pacf(values, name, lags=30)

### Your guesses

Write your guesses here before revealing the answer:

- Series A:
- Series B:
- Series C:

Explain your reasoning using the ACF and PACF patterns.

In [ ]:
answers = pd.DataFrame({
    "series": ["Series A", "Series B", "Series C"],
    "true_model": ["MA(1), theta=-0.65", "AR(1), phi=0.80", "ARMA(1,1), phi=0.55, theta=0.70"],
    "main_signature": [
        "ACF tends to cut off quickly; negative lag-1 correlation",
        "ACF decays slowly; PACF has strong early cutoff behavior",
        "ACF and PACF both tend to decay"
    ]
})
answers

## 15. Mini-project

Choose one of the following mini-projects.

### Option A: AR(1) parameter sweep

Simulate AR(1) models for

$$
\phi \in \{-0.9, -0.5, 0, 0.5, 0.9\}.
$$

For each model:

1. Plot the series.
2. Plot the ACF.
3. Compare the sample ACF with $\rho(h)=\phi^h$.
4. Write a paragraph explaining how the sign and magnitude of $\phi$ change the behavior.

### Option B: MA(1) parameter sweep

Simulate MA(1) models for

$$
\theta \in \{-0.9, -0.5, 0.5, 0.9\}.
$$

For each model:

1. Plot the series.
2. Plot the ACF.
3. Compare the sample lag-1 ACF with $\theta/(1+\theta^2)$.
4. Explain why the ACF should be approximately zero after lag 1.

### Option C: Root-condition experiment

Pick three AR(2) models. For each one:

1. Compute the roots of $1-\phi_1 z - \phi_2 z^2$.
2. Decide whether the AR part is causal.
3. Simulate the model.
4. Explain whether the time plot agrees with the root condition.

In [ ]:
# Starter code for Option A
phis_to_try = [-0.9, -0.5, 0.0, 0.5, 0.9]

for phi_value in phis_to_try:
    x_temp, _ = simulate_ar1(phi=phi_value, n=300, burnin=300, seed=500)
    plot_series_and_acf(x_temp, f"AR(1), phi={phi_value}", max_lag=25)

## 16. AI-assisted study prompts

Use AI tools as a study assistant, not as a replacement for your reasoning. After you complete the lab, try the following prompts.

### Prompt 1: Explain ACF patterns

> I simulated an AR(1), an MA(1), and an ARMA(1,1) process. Explain why the AR(1) ACF decays geometrically, why the MA(1) ACF cuts off after lag 1, and why the ARMA ACF is harder to identify visually.

### Prompt 2: Check my root-condition reasoning

> For the model $(1-0.7B)X_t=(1+0.5B)W_t$, I computed the AR root and MA root. Check whether my conclusion about causality and invertibility is correct. Explain the root condition step by step.

### Prompt 3: Debug my simulation

> I wrote Python code to simulate an ARMA(1,1) process, but the sample ACF does not look reasonable. Here is my code: [paste code]. Please check whether I used the noise terms and burn-in correctly.

### Responsible use reminder

When using AI, always verify:

1. Does the model equation match the code?
2. Are indices correct?
3. Is a burn-in period used for recursive models?
4. Are stationarity and causality assumptions stated clearly?
5. Are interpretations supported by plots and calculations?

## 17. Lab checklist

Before finishing, make sure you can do the following without looking at the solutions:

- [ ] Define white noise.
- [ ] Simulate AR(1), MA(1), and ARMA(1,1) models from scratch.
- [ ] Explain the difference between dependence on past values and dependence on past shocks.
- [ ] Compute a sample ACF from scratch.
- [ ] State the theoretical ACF for AR(1).
- [ ] State the theoretical ACF cutoff property for MA(1).
- [ ] Explain why AR roots outside the unit circle are important.
- [ ] Explain why MA roots outside the unit circle are important.
- [ ] Interpret impulse-response weights.
- [ ] Use plots to distinguish AR-like, MA-like, and ARMA-like behavior.